In [1]:
import os
import sys

# Python project
path = r"C:\MARL_AnyLogic\thesis"
sys.path.insert(0, path)

# Java 17 cho AnyLogic/Alpyne
java_home = r"C:\Program Files\AnyLogic 8.9 Personal Learning Edition\jre"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = os.path.join(java_home, "bin") + os.pathsep + os.environ["PATH"]

from thesis.utils.utils import setup_ray, save, load, Experiment

setup_ray(path=path, unidirectional=False, seed=69)

print("RAY INITIALIZED SUCCESSFULLY!")

2026-10-13 22:37:53,538	INFO worker.py:1528 -- Started a local Ray instance.


RAY INITIALIZED SUCCESSFULLY!


In [2]:
env_args = dict(
    fleetsize = 6,
    fleetsize_upper = 10,
    max_fleetsize = 20,    
    pseudo_routing = False,
    pseudo_dispatcher = True,
    pseudo_dispatcher_clever = False,
    #pseudo_dispatcher_distance = 0.3,
    routing_agent_death= True,
    death_on_target = True,
    transform_dispatching_partobs = True,
    direction_reward = -0.1,
    sim_config = dict(
        dispatch = True,
        routing_ma = True,
        dispatching_ma = True,
        reward_reached_target = 10,
        #reward_reached_target_by_time = True, 
        reward_wrong_target = -1,
        reward_removed_for_block = -20, 
        reward_target_distance = 0,
        reward_invalid= -0.1,
        reward_duration = -0.5,
        reward_pass = 0.2,
        block_timeout = 30,
        #reward_accepted_in_station = 1,
        #reward_declined_in_station = -1,
        routing_interval = 2,
        dispatching_interval=360,
        io_quote = 0.99,
        availability = 0.95,
        mttr = 5,
        remove_all_blocked = False,
    )
)

In [3]:
agv_model = dict(
    model = dict(
        custom_model = "gnn_model",
        #custom_action_dist="MAActionDistribution",
        custom_model_config = dict(
            embed_dim=32,
            with_action_mask=False,
            with_agvs=True,
            with_stations = False,
            position_embedd_dim = 0,
            ff_embedd_dim = 4,
            env_type = "matrix",
            n_convolutions = 2,
        )
    )
)
dispatcher_model = dict(
    model = dict(
        custom_model = "lin_model",
        #custom_action_dist="MAActionDistribution",
        custom_model_config = dict(
            embed_dim=64,
            with_action_mask=False,
            with_agvs=True,
            with_stations = True,
            position_embedd_dim = 0,
            ff_embedd_dim = 4,
        )
    )
)

In [ ]:
exp = Experiment("matrix_routing")
for seed in [43]:
    exp.experiment(
        path = path,
        env_args = env_args, 
        agv_model = agv_model,
        dispatcher_model=dispatcher_model,
        run_name="DQN_ROUTING_2NEV", 
        env = "matrix",
        algo = "dqn",
        n_intervals =0,
        #train_agv = False, 
        backup_interval=1,
        batch_size=128, #apex + gnn: 50
        seed = seed,
        algo_params = {"grad_clip": 1, "gamma":0.9, "exploration_config":{"warmup_timesteps": 0,"epsilon_timesteps": 200000,"final_epsilon": 0.02,"initial_epsilon": 1,"type": "EpsilonGreedy"}},# "exploration_config": {"type": "Curiosity", "sub_exploration": {"type": "StochasticSampling"}, "eta": 0.1}},
        lr = 1e-3,
        n_envs=2,
        load_agv= r"C:\MARL_AnyLogic\thesis\models\matrix_routing\DQN_ROUTING_TEST_6_20_2026-10-13_21-44-16\checkpoint_000012",
        
    )

2026-10-13 22:39:58,247	INFO simple_q.py:307 -- In multi-agent mode, policies will be optimized sequentially by the multi-GPU optimizer. Consider setting `simple_optimizer=True` if this doesn't work for you.
2026-10-13 22:39:58,250	INFO algorithm.py:457 -- Current log_level is ERROR. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
c:\MARL_AnyLogic\.venv39\lib\site-packages\alpyne\client\utils.py:124: UserWarning: Unzipping to temporary directory (C:\Users\MINHTR~1\AppData\Local\Temp\alpyne_94442_6t2bgtyl)
  warn(f"Unzipping to temporary directory ({tmp_dir})")
2026-10-13 22:40:07,593	WARNING util.py:66 -- Install gputil for GPU system monitoring.


ValueError: Could not recover from checkpoint as it does not exist on local disk and was not available on cloud storage or another Ray node. Got checkpoint path: C:\MARL_AnyLogic	hesis\models\matrix_routing\DQN_ROUTING_TEST_6_20_2026-10-13_21-44-16\checkpoint_000012 and IP None

In [12]:
from pathlib import Path

checkpoints = sorted(
    Path(exp.checkpoint_dir).glob("checkpoint_*")
)

assert checkpoints, "Chưa tìm thấy checkpoint!"

old_checkpoint = str(checkpoints[-1])

print("CHECKPOINT CU:", old_checkpoint)
print("ITERATION:", exp.trainer.iteration)

CHECKPOINT CU: C:\MARL_AnyLogic\thesis\models\matrix_routing\DQN_ROUTING_TEST_6_20_2026-10-13_21-44-16\checkpoint_000012
ITERATION: 12


In [13]:
exp.trainer.stop()

import ray
ray.shutdown()

In [7]:
import time

print("BAT DAU TEST DQN VOI 2 ENV...", flush=True)

t0 = time.time()

result = exp.trainer.train()

elapsed = time.time() - t0

print("===== KET QUA 2 ENV =====")
print("Iteration:", result.get("training_iteration"))
print("Timesteps:", result.get("timesteps_total"))
print("Episodes:", result.get("episodes_total"))
print("Mean Reward:", result.get("episode_reward_mean"))
print("Thoi gian:", round(elapsed, 2), "giay")

# Luu checkpoint sau khi train
checkpoint = exp.trainer.save(exp.checkpoint_dir)
print("Checkpoint:", checkpoint)

BAT DAU TEST DQN VOI 2 ENV...
===== KET QUA 2 ENV =====
Iteration: 13
Timesteps: 13000
Episodes: 1
Mean Reward: nan
Thoi gian: 62.13 giay


AttributeError: 'Experiment' object has no attribute 'checkpoint_dir'

In [8]:
from pathlib import Path

root = Path(path) / "models" / "matrix_routing"

folders = sorted(
    root.glob("DQN_ROUTING_2ENV_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

assert folders, "Không tìm thấy thư mục DQN_ROUTING_2ENV!"

exp.checkpoint_dir = str(folders[0])
exp.backup_interval = 1

print("Iteration hiện tại:", exp.trainer.iteration)
print("Thư mục lưu:", exp.checkpoint_dir)

checkpoint = exp.trainer.save(exp.checkpoint_dir)

print("ĐÃ LƯU CHECKPOINT:", checkpoint)

Iteration hiện tại: 13
Thư mục lưu: C:\MARL_AnyLogic\thesis\models\matrix_routing\DQN_ROUTING_2ENV_6_20_2026-10-13_22-39-58
ĐÃ LƯU CHECKPOINT: C:\MARL_AnyLogic\thesis\models\matrix_routing\DQN_ROUTING_2ENV_6_20_2026-10-13_22-39-58\checkpoint_000013


In [5]:
#load(exp.trainer, "agv", "../../models/trained_routing")

In [6]:
save(exp.trainer, "agv", "../../models/final")

In [7]:
#exp.keep_training(4)

In [8]:
#exp.keep_training(2)